# 04 — Classical Baseline Models

This notebook demonstrates the five **classical baseline** models used for comparison against the Quantum Autoencoder.  
Each model is trained on normal data and produces anomaly scores for unseen instances.

| Model | Type | Parameters |
|-------|------|------------|
| MatchedAutoencoder | PyTorch AE (6→4→6) | 100 |
| FullAutoencoder | PyTorch AE (20→12→6→12→20) | 3 006 |
| LSTMAutoencoder | LSTM seq2seq | 3 029 |
| IsolationForest | Ensemble (scikit-learn) | n/a |
| OneClassSVM | Kernel SVM (scikit-learn) | n/a |

## 1. Data Preparation

We prepare training data the same way as the QAE notebook — load a normal instance, run `FeatureEngineer`, and limit to 50 windows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qml.samarone_junior.loaders import ThreeWLoader, FeatureEngineer
from qml.samarone_junior.models import (
    MatchedAutoencoder,
    FullAutoencoder,
    LSTMAutoencoder,
    IsolationForestDetector,
    OneClassSVMDetector,
)

loader = ThreeWLoader(data_path="../../data/samarone_junior/3w")
fe = FeatureEngineer(n_components=6, window_size=128, stride=64)

instances_normal = loader.list_instances(0)
if instances_normal:
    df = pd.read_parquet(instances_normal[0])
    raw = df[ThreeWLoader.SENSORS].dropna().values
    windows = fe.extract_windows(raw)
    features = fe.compute_features(windows)
    X = fe.fit_transform(features)
else:
    X = np.random.uniform(0, np.pi, (100, 6))

X_train = X[:50]
print(f"Training shape: {X_train.shape}")

## 2. Train Each Baseline

We instantiate each model, train on the normal windows, and store anomaly scores on the training set.

In [ ]:
models = {
    "MatchedAutoencoder": MatchedAutoencoder(seed=42),
    "FullAutoencoder": FullAutoencoder(seed=42),
    "IsolationForest": IsolationForestDetector(seed=42),
    "OneClassSVM": OneClassSVMDetector(seed=42),
}

all_scores = {}
for name, model in models.items():
    losses = model.fit(X_train, n_epochs=5)
    scores = model.anomaly_scores(X_train)
    all_scores[name] = scores
    print(f"{name:25s}  mean_score={np.mean(scores):.4f}  std={np.std(scores):.4f}")

## 3. LSTM Autoencoder (Sequence-Based)

The `LSTMAutoencoder` operates on raw 3D window arrays (n_windows × window_size × n_sensors) rather than flattened PCA features. We demonstrate it separately with a small window.

In [ ]:
# LSTM works on raw windowed data (3D)
if instances_normal:
    windows_3d = fe.extract_windows(raw)[:50]
else:
    windows_3d = np.random.randn(50, 128, 5)

lstm_model = LSTMAutoencoder(seed=42, hidden_size=16)
lstm_losses = lstm_model.fit(windows_3d, n_epochs=5)
lstm_scores = lstm_model.anomaly_scores(windows_3d)
all_scores["LSTMAutoencoder"] = lstm_scores
print(f"LSTM  mean_score={np.mean(lstm_scores):.4f}  std={np.std(lstm_scores):.4f}")

## 4. Score Distribution Comparison

We visualise the anomaly score distributions across all baseline models. On normal training data, most scores should cluster low.

In [ ]:
fig, axes = plt.subplots(1, len(all_scores), figsize=(15, 3), sharey=True)
for ax, (name, scores) in zip(axes, all_scores.items()):
    ax.hist(scores, bins=20, alpha=0.7, color="steelblue")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Score")
axes[0].set_ylabel("Count")
fig.suptitle("Anomaly Score Distributions (Normal Training Data)", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Summary

- **MatchedAutoencoder** and **FullAutoencoder** are reconstruction-based — scores are MSE between input and output.
- **IsolationForest** uses path-length isolation — scores are inverted so higher = more anomalous.
- **OneClassSVM** uses kernel-based decision boundaries — scores are distance from the boundary.
- **LSTMAutoencoder** operates on sequential (3D) data — useful for capturing temporal patterns.
- Full cross-validated comparison across all event classes is in notebook 05.